# Part 3: image agents with measured evaluation

This is the revised implementation, not the earlier baseline. Run it in a fresh GPU kernel,
one notebook at a time. A nominal 16-GB GPU is the lab target. The RTX 4050 uses a CPU-offloaded
low-VRAM profile, which must pass the real preflight on your own PC. A cold setup needs internet,
roughly 40 GB free disk overall and enough host RAM for offload. No hosted model API is used.

The default is a full run. `LAB2_SMOKE_ONLY=1` explicitly skips model-backed measurements;
it is a developer check, never a completed experiment. Real human ratings are never generated.
Device-memory sampling covers the actual subprocess stages but is not proof that an RTX 4090
is a 16-GB GPU. Keep the printed hardware identity and all outputs.

Before leaving the computer, run through the real GPU preflight and confirm it passes.
Matching completed items resume from atomic checkpoints. Changed code, data or hardware starts
a separate checkpoint namespace; no old result is relabeled. Keep the machine awake and the
local GPU runtime running. Closing the local server or rebooting stops execution.

Your Part 2 training is reused, not repeated. Private adapter download asks for a read token securely before the expensive stages. Never put the token in this notebook or chat.

In [1]:
from pathlib import Path
import os, subprocess, sys, platform, json, time

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "tools/setup_runtime.py").exists()), None)
if ROOT is None:
    if os.environ.get("LAB2_ROOT"):
        ROOT = Path(os.environ["LAB2_ROOT"]).expanduser().resolve()
    elif Path("/content").exists():
        ROOT = Path("/content/Lab-2")
    elif Path("/kaggle/working").exists():
        ROOT = Path("/kaggle/working/Lab-2")
    else:
        ROOT = Path.home() / "DATA266" / "Lab-2"
    if not ROOT.exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").exists():
        raise RuntimeError(f"{ROOT} is an older/incomplete checkout. Update it with git pull, then restart this notebook.")
print("Notebook host:", platform.python_version(), "repository:", ROOT)
print("Mode:", "SMOKE ONLY: no full training or model-backed evaluation" if SMOKE_ONLY else "FULL RUN")
# Refresh a clean main checkout, but never overwrite local edits or reset a branch.
if not SMOKE_ONLY and os.environ.get("LAB2_UPDATE_REPOSITORY", "1") == "1":
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
    if branch == "main" and not dirty:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    else:
        print("Kept existing checkout: local edits or a non-main branch. No files were reset.")
if "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
    # A freshly downloaded notebook must not silently execute the old broken installer.
    previous_root = ROOT
    ROOT = ROOT.with_name(ROOT.name + "-notebook-v2")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").is_file() or "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
        raise RuntimeError(f"Update the checkout at {ROOT} before running. The original checkout was preserved.")
    print("Using repaired code at", ROOT, "; the older checkout and its results remain at", previous_root)
subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                "--device", "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")], check=True)
requested_device = "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")
import shutil
use_cuda = requested_device == "cuda" or (requested_device == "auto" and shutil.which("nvidia-smi") is not None)
runtime_file = ROOT / (".lab2_runtime_cuda.json" if use_cuda else ".lab2_runtime_cpu.json")
PYTHON = Path(json.loads(runtime_file.read_text())["python"])
SEED = 266
print("All remaining commands use:", PYTHON)

def run_script(relative, *arguments):
    command = [str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f"{Path(relative).name} stopped (exit {code}). Full diagnostic log: {log}")

# Permit first-time downloads during preflight, even after restarting this notebook.
if not SMOKE_ONLY:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
# Monitor actual device memory in each subprocess. A larger GPU is not relabeled as a T4.
_run_stage = run_script
def run_script(relative, *arguments):
    record = ROOT / "runtime_checks" / (Path(relative).stem + "_" + str(time.time_ns()) + ".memory.json")
    _run_stage("tools/monitored_run.py", "--record", record, "--", PYTHON, ROOT / relative, *arguments)


Notebook host: 3.12.6 repository: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2
Mode: SMOKE ONLY: no full training or model-backed evaluation


{
  "commit": "0aad9eb",
  "working_tree_dirty": true,
  "python": "3.12.6",
  "seed": 266,
  "executable": "/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python",
  "versions": {
    "torch": "2.5.1",
    "torchvision": "0.20.1",
    "diffusers": "0.31.0",
    "transformers": "4.46.2",
    "peft": "0.13.2",
    "datasets": "3.1.0",
    "huggingface-hub": "0.26.2"
  },
  "cuda_available": false,
  "gpu": "cpu"
}
PASS: core imports, matching torchvision kernels and device preflight


LAB2_PYTHON=/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python
All remaining commands use: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python


## Offline regression tests
No mocked output is counted as task success. The full model-backed tests below are separate.

In [2]:
run_script("tools/test_lab_completion.py")
run_script("tools/test_structured_responses.py")
run_script("tools/test_overnight_resume.py")
run_script("part3_agents/src/smoke_test.py")

Running monitored_run.py


test_actual_tiny_qwen_multimodal_forward (__main__.TestAgentSafety.test_actual_tiny_qwen_multimodal_forward) ... `Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


ok


test_image_instructions_are_data_and_refused (__main__.TestAgentSafety.test_image_instructions_are_data_and_refused) ... ok


test_instruction_normalization (__main__.TestAgentSafety.test_instruction_normalization) ... ok


test_json_decode_fails_closed (__main__.TestAgentSafety.test_json_decode_fails_closed) ... ok


... (63 outputs left out)

Measured stage record: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/runtime_checks/test_overnight_resume_1794273740065062000.memory.json


Running monitored_run.py


PASS: safe routing, corrupt files, real image validation, bounded revisions, unique outputs and error recovery


Measured stage record: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/runtime_checks/smoke_test_1794273744525742000.memory.json


## Real inputs and the pinned Part 2 Hub adapter
Three actual held-out Naruto images are copied locally; synthetic instruction overlays and broken files are explicitly safety fixtures. This is a narrow anime-domain test set, not proof of broad photographic performance. Input hashes are saved. The adapter is fetched from its verified Hugging Face commit, not silently replaced by local/base weights.

In [3]:
P3 = ROOT / "part3_agents"
RUN_DIR = P3 / "outputs/v2" / ("smoke" if SMOKE_ONLY else "runs") / time.strftime("%Y%m%d_%H%M%S")
if SMOKE_ONLY:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    (P3 / "outputs/v2/latest_smoke.json").write_text(json.dumps({"folder": str(RUN_DIR)}))
INPUTS = P3 / "outputs/v2/inputs/inputs.json"
ADAPTER = P3 / "outputs/hub_adapter"
if SMOKE_ONLY:
    print("SKIPPED: real image preparation, private adapter download and full models.")
else:
    run_script("tools/run_checkpoint.py", "--select-run", "part3_agents")
    RUN_DIR = Path(json.loads((P3 / "outputs/v2/latest_run.json").read_text())["folder"])
    from getpass import getpass
    temporary_token = False
    import hashlib
    expected_adapter = json.loads((ROOT / "part2_diffusion/evidence/hub_links.json").read_text())
    already_downloaded = ((ADAPTER / "hub_provenance.json").is_file()
        and (ADAPTER / "pytorch_lora_weights.safetensors").is_file()
        and hashlib.sha256((ADAPTER / "pytorch_lora_weights.safetensors").read_bytes()).hexdigest()
            == expected_adapter["run1_adapter_sha256"])
    if not os.environ.get("HF_TOKEN") and not already_downloaded:
        os.environ["HF_TOKEN"] = getpass("Hugging Face read token for your private adapter (hidden): ")
        temporary_token = True
    try:
        run_script("part3_agents/src/prepare_evaluation.py", "--download-adapter")
    finally:
        if temporary_token:
            os.environ.pop("HF_TOKEN", None)
    print("Adapter provenance:", (ADAPTER / "hub_provenance.json").read_text())

SKIPPED: real image preparation, private adapter download and full models.


## Actual GPU preflight
Before the long comparison, exercise the real VLM, critic, style pipeline, instruction editor and domain adapter sequentially. Two-step preflight images are diagnostic fixtures, not final generated results. Confirm `PREFLIGHT PASSED` on this RTX 4050 before leaving it overnight.

In [4]:
if SMOKE_ONLY:
    print("SKIPPED: actual GPU preflight. The local CPU smoke does not certify RTX 4050 readiness.")
else:
    run_script("tools/overnight_preflight.py", "--part", "part3_agents")
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

SKIPPED: actual GPU preflight. The local CPU smoke does not certify RTX 4050 readiness.


## Run both systems
Style changes use SD 1.5 img2img; edits use InstructPix2Pix. Domain requests use the pinned Part 2 adapter. Qwen2-VL transcribes image text, answers visual questions, rewrites prompts and critiques original versus generated images. The VLM is unloaded before diffusion to bound concurrent VRAM. Both configurations use the same inputs, prompts and initial seed; revisions are capped at two.

In [5]:
if SMOKE_ONLY:
    print("SKIPPED: 30 real model-backed scenario executions.")
else:
    for filename, without in [("scenario_traces_no_critique.json", True), ("scenario_traces.json", False)]:
        arguments = ["--inputs", INPUTS, "--output", RUN_DIR / filename,
                     "--enable-models", "--lora-path", ADAPTER]
        if without:
            arguments.append("--no-critique")
        run_script("part3_agents/src/run_scenarios.py", *arguments)
    from IPython.display import Image, display
    for filename in ["scenario_traces_no_critique.json", "scenario_traces.json"]:
        traces = json.loads((RUN_DIR / filename).read_text())
        for trace in traces:
            print(filename, trace["scenario_id"], trace["request"], trace["decision"], trace["message"])
            print("Caption:", trace["caption"], "Answer:", trace["visual_answer"])
            print("Critique:", trace["critique"], "Seconds:", trace["seconds"])
            if trace["output_path"]:
                display(Image(filename=trace["output_path"], width=384))

SKIPPED: 30 real model-backed scenario executions.


## Real image-based safety evaluation
The ten specified cases include instructions in actual image pixels, unsafe requests, a corrupt file, an empty file and missing input. A model/runtime error is not accepted as a successful safe refusal. Baseline and critique results are measured separately.

In [6]:
if SMOKE_ONLY:
    print("SKIPPED: image-based attack-success measurement.")
else:
    for filename, without in [("red_team_results_no_critique.json", True), ("red_team_results.json", False)]:
        arguments = ["--inputs", INPUTS, "--enable-models", "--output", RUN_DIR / filename]
        if without:
            arguments.append("--no-critique")
        run_script("part3_agents/src/run_red_team.py", *arguments)
    run_script("part3_agents/src/evaluate_agents.py", "--folder", RUN_DIR)
    print((RUN_DIR / "evaluation.json").read_text())

SKIPPED: image-based attack-success measurement.


## Independent team judgments
Each member rates every baseline and final scenario against its saved rubric. Fill `human_ratings_template.json`, save as `human_ratings.json`, and record how disagreements were resolved. Until then task-success and critic-agreement values stay pending. Automatic critic acceptance is never substituted for a human pass.

In [7]:
run_script("tools/prepare_rating_forms.py")
if not SMOKE_ONLY:
    run_script("part3_agents/src/evaluate_agents.py", "--folder", RUN_DIR)
run_script("tools/final_submission_audit.py")
if not SMOKE_ONLY:
    run_script("tools/build_combined_report.py")
else:
    print("SKIPPED: rebuilding the submission report from a developer smoke run.")
run_script("tools/export_lab_results.py", "--part", "part3_agents")

Running monitored_run.py


Prepared blank forms: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/docs/rating_forms


No observed score or human judgment has been generated. The synthetic example is not an input rating file.


Measured stage record: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/runtime_checks/prepare_rating_forms_1794273744645099000.memory.json


Running monitored_run.py


Part 2 training comparison


... (36 outputs left out)

Result ZIP created on the runtime filesystem: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/output/exports/part3_agents_20261110T012224930430Z.zip


Size MiB: 15.45


Save the executed notebook first and rerun this export cell to include its latest outputs. A local/Docker Colab runtime stores this on the GPU machine, not automatically in browser Downloads.


Measured stage record: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/runtime_checks/export_lab_results_1794273744867685000.memory.json


## Optional upload action after reviewing results
The generated images belong on Hugging Face, not in Git history. This action stays disabled until you review the actual outputs and permitted sharing. Enter a write token securely, never in source code. Only final safe outputs and their metadata are uploaded privately; arrange grader access separately.

In [8]:
UPLOAD_PART3_IMAGES = False
RIGHTS_REVIEWED = False
if UPLOAD_PART3_IMAGES and not SMOKE_ONLY:
    if not RIGHTS_REVIEWED:
        raise ValueError("Review educational sharing rights before upload.")
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face write token (hidden): ")
    try:
        run_script("part3_agents/src/publish_results.py", "--folder", RUN_DIR,
                   "--namespace", "denishatank", "--rights-reviewed")
    finally:
        os.environ.pop("HF_TOKEN", None)
else:
    print("Part 3 image upload not requested; no upload is claimed.")

Part 3 image upload not requested; no upload is claimed.


## Gradio demo
Run this cell when you are ready to test the interactive demo. It stays local and does not expose a public server. Show one caption/question, one generated edit/style request and one safe failure; both team members need to rehearse. The notebook does not pretend a disabled app was demonstrated.

In [9]:
LAUNCH_APP = False
if LAUNCH_APP and not SMOKE_ONLY:
    app_process = subprocess.Popen([str(PYTHON), str(P3 / "src/app.py")], cwd=ROOT)
    print("Local demo started. Open the local address printed by Gradio; stop app_process when finished.")
else:
    print("Interactive demo not launched. Enable this cell when ready to rehearse.")

Interactive demo not launched. Enable this cell when ready to rehearse.


## Download one ZIP with all available lab results

Run this cell after the overnight work. It bundles all available results for Parts 1, 2 and 3, executed notebooks, source/configuration, reports, adapters, generated images and diagnostic logs. Base-model downloads, environments, original datasets and old archives are excluded. Missing files are listed honestly in `BUNDLE_MANIFEST.json`.

The ZIP transfers through the Colab notebook connection, including a local kernel, without a localhost hyperlink. A native Windows/Mac runtime also copies it into its accessible Downloads folder. Browser location follows your download settings. Keep the kernel connected until transfer finishes. If the run fails, you can execute only this export cell to send the partial evidence; no GPU work is repeated.

In [10]:
from pathlib import Path
import os, sys, subprocess, json, importlib.util

if "ROOT" not in globals():
    candidates = [Path.cwd(), *Path.cwd().parents,
        Path(os.environ.get("LAB2_ROOT", str(Path.home() / "DATA266/Lab-2"))),
        Path("/content/Lab-2"), Path("/kaggle/working/Lab-2")]
    ROOT = next((p.resolve() for p in candidates if (p / "tools/export_lab_results.py").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Set LAB2_ROOT to your existing Lab-2 folder. Do not rerun training just to export files.")

# This script only collects saved files. It does not install packages, train, or generate.
subprocess.run([sys.executable, str(ROOT / "tools/export_lab_results.py"), "--all"], cwd=ROOT, check=True)
bundle = json.loads((ROOT / "output/exports/latest_all_results.json").read_text())
print("Upload this ZIP in our chat:", bundle["archive"])
print("ZIP SHA256:", bundle["sha256"])
if globals().get("SMOKE_ONLY", False):
    print("CPU smoke: bundle verified, browser download intentionally not triggered. These are not new GPU results.")
else:
    spec = importlib.util.spec_from_file_location("lab2_download_results", ROOT / "tools/download_results.py")
    downloader = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(downloader)
    downloader.offer_download(bundle["archive"], auto_start=True)

ALL RESULTS ZIP: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/output/exports/Lab2_All_Results_20261110T012225591153Z.zip
Size MiB: 142.2 Files: 437
SHA256: fbf479157c9f98881f87f285a42ed86d5cfcffb553b2bd899f2b8d035c385dde
Missing on THIS runtime (see manifest): part1_rag/outputs/v2/evaluation/evaluation.json, part3_agents/outputs/v2/latest_run.json, CONTRIBUTIONS.md, links.txt
No training or generation was repeated. Review the archive before sharing; common token patterns were redacted.
Upload this ZIP in our chat: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/output/exports/Lab2_All_Results_20261110T012225591153Z.zip
ZIP SHA256: fbf479157c9f98881f87f285a42ed86d5cfcffb553b2bd899f2b8d035c385dde
CPU smoke: bundle verified, browser download intentionally not triggered. These are not new GPU results.
